In [1]:
import pandas as pd
import numpy as np
import os

import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow import keras

### Neural networks can find groups or pattern and pull out meanings.

# How do we choose layers, neurons and hyperparamaters?
## 1) Use training performance to guide your decisions
#### => High accuracy on training but not validation (Overfil) : Reduce # of paramaters
#### => Low accuracy on training :  Reduce # of paramaters

## 2) Automatically search for the best hyperparamaters with grid search (learning rate, batch size, optimizer, dropout, etc.) 

# Why do we need an activation function?

## 1) To introduce non-linearity into our neural net calculations
## 2) This is what allows us to fit more complex data & compute some really exciting things.

###### Relu helps avoid vanishing gradient problem 

1. Why use the "Information Funnel" (128 → 64 → 32)?
Think of a neural network like a detective agency.

The First Layer (128 neurons): This layer looks at the raw data. It needs a lot of "eyes" (neurons) to catch every tiny detail, edge, and shadow.

The Middle Layer (64 neurons): It takes the details from the first layer and starts combining them. It doesn't need to see the raw pixels anymore; it only needs to see the "concepts" (like "this is a circle" or "this is a line").

The Final Hidden Layer (32 neurons): It condenses those concepts into high-level features (like "this is a face").
The Goal: You are forcing the model to compress the information. If you kept 128 neurons in every layer, the model might just "memorize" the noise in the data. By squeezing the data through a smaller "neck," you force it to keep only the most important patterns.


2. Other "Unspoken Rules" for LayersRule of Powers of TwoYou’ll notice people almost always use 32, 64, 128, 256, or 512.The "Why": This isn't for the AI's "brain"—it's for your Computer's hardware. Modern GPUs and CPUs are optimized to process data in chunks of $2^n$. Using these numbers makes your training run significantly faster because the memory alignment is perfect.The "Bottleneck" RuleNever make a hidden layer smaller than your output layer.

The "Why": If you are trying to classify 10 different types of clothes, but you have a hidden layer with only 3 neurons, you have created a "bottleneck." You are throwing away too much information before the model can make a final decision.

3. Start with "ReLu", End with "Softmax" or "Sigmoid"Hidden Layers: Use ReLU (Rectified Linear Unit). It’s fast and prevents the "vanishing gradient" problem where the model stops learning.

Output Layer: 
* Use Sigmoid if it's a Yes/No question (Binary).Use Softmax if you are choosing between multiple categories (like Cat vs Dog vs Bird), because it makes the total probability add up to 100%.Don't Go Too Deep Too FastFor most beginner projects (like recognizing handwriting or basic CSV data), two to three hidden layers are plenty. Adding 10 layers sounds like it would make the model "smarter," but it actually makes it much harder to train and prone to "overfitting."



In [2]:
os.getcwd()

'C:\\Users\\newta\\OneDrive\\Desktop\\REPO\\Python-Frameworks-Tutorials\\INTRO_TO_NEURAL_NETWORKS'

In [3]:
os.listdir('.')

['.ipynb_checkpoints',
 'clusters',
 'clusters_two_categories',
 'complex',
 'Images',
 'linear',
 'Neural Net.ipynb',
 'quadratic']

In [4]:
os.listdir('./linear')

['data', 'figure.png', 'network_linear.py']

In [5]:
os.listdir('./linear/data')

['test.csv', 'train.csv']

# Linear

In [6]:
train_df = pd.read_csv('./linear/data/train.csv')

In [7]:
train_df.head()

,x,y,color
0,1.146728,2.233629,0.0
1,3.676886,4.520687,0.0
2,0.730671,1.426260,0.0
3,1.950790,3.145987,0.0
4,4.323010,5.320534,0.0


In [8]:
# It is important to shuffle the data, not like att 0 color at the topm, and all 1 at the bottom.
np.random.shuffle(train_df.values)

In [9]:
train_df.head(15)

,x,y,color
0,3.125637,2.596340,1.0
1,1.410599,0.430899,1.0
2,4.167108,4.717616,0.0
3,2.183101,0.956210,1.0
4,1.536210,0.245647,1.0
5,1.242295,2.311984,0.0
6,0.073968,0.798434,0.0
7,1.633360,0.850780,1.0
8,4.788419,5.533048,0.0
9,4.320130,5.349706,0.0


In [10]:
model = keras.Sequential([
    keras.layers.Dense(4,input_shape=(2,), activation = 'relu'),
    keras.layers.Dense(2)
])

C:\Users\newta\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1. Correcting the MathYou are exactly right about the error. Let's look at your examples:Scenario A: from_logits=TrueThe loss function takes your raw numbers (5.2, -1.3), internally runs the Softmax formula to turn them into probabilities (0.998, 0.002), and then does the math:$$Loss = -(0 \cdot \log(0.998) + 1 \cdot \log(0.002))$$This works perfectly because $\log$ of a positive decimal is a valid number.Scenario B: from_logits=False (The Error)The loss function assumes the numbers you sent are already probabilities. It tries to calculate:$$Loss = -(0 \cdot \log(5.2) + 1 \cdot \log(-1.3))$$The Crash: You cannot take the logarithm of a negative number ($-1.3$). This is why your model would return NaN (Not a Number) or throw an error.2. Does it automatically apply Softmax?Yes and No—and this is a very important distinction:During Training: Yes. When calculating the Loss, Keras automatically applies Softmax internally to figure out the error and update the weights.During Prediction (model.predict): No. This is the "catch." If you use from_logits=True and then later try to use your model to predict a new value, model.predict(new_data) will still give you raw numbers like [5.2, -1.3]. It won't look like probabilities unless you manually wrap the output in a Softmax function or add the layer back in later.3. Summary of the WorkflowWhen you use your current model with from_logits=True:Last Layer: Spits out raw "Logits" (e.g., 10.5, -2.3).Loss Function: Sees from_logits=True, converts those to probabilities (e.g., 0.99, 0.01), and calculates the error.Optimizer: Uses that error to fix the weights.

In [11]:
model.compile(optimizer = 'adam', 
              loss = keras.losses.SparseCategoricalCrossentropy(from_logits = True),
             metrics = ['accuracy'])

In [12]:
print(type(train_df.x.values))

<class 'numpy.ndarray'>


In [13]:
x = np.column_stack((train_df.x.values, train_df.y.values))

In [14]:
type(x)

numpy.ndarray

In [15]:
x[0:5]

array([[3.12563717, 2.59633997],
       [1.4105991 , 0.43089913],
       [4.16710826, 4.71761608],
       [2.18310081, 0.95620952],
       [1.53620985, 0.24564726]])

In [16]:
model.fit(x, train_df.color.values, batch_size = 4)

1000/1000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.3740 - loss: 0.7182


#### Evaluating the test data

In [17]:
test_df = pd.read_csv('./linear/data/test.csv')

In [18]:
# np.random.shuffle(test_df.color.values)

In [19]:
test_df.head()

,x,y,color
0,2.684292,3.867108,0.0
1,2.707883,4.002614,0.0
2,2.705905,3.859686,0.0
3,4.536191,5.240051,0.0
4,3.656068,4.461771,0.0


In [20]:
test_x = np.column_stack((test_df.x.values, test_df.y.values))

In [21]:
model.evaluate(test_x, test_df.color.values)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4930 - loss: 0.6934


[0.6933919191360474, 0.49300000071525574]

# Quadratic

In [22]:
os.listdir('./quadratic/data')

['test.csv', 'train.csv']

In [23]:
train_df = pd.read_csv('./quadratic/data/train.csv')

In [24]:
train_df.head()

,x,y,color
0,-4.956506,25.706334,0.0
1,2.897218,10.359784,0.0
2,-4.488273,22.113311,0.0
3,3.823152,15.665060,0.0
4,4.425201,21.118726,0.0


In [25]:
np.random.shuffle(train_df.values)

In [26]:
train_df.head()

,x,y,color
0,3.649095,14.604536,0.0
1,0.917710,2.104694,0.0
2,-0.373653,-1.169612,1.0
3,-1.740039,4.381882,0.0
4,-2.677594,5.770459,1.0


In [27]:
# added more neurons in the first layer
# also add dropout of 20% 
model = keras.Sequential([
    keras.layers.Dense(32,input_shape=(2,), activation = 'relu'), 
   # keras.layers.Dropout(0.2), : Does not help
     keras.layers.Dense(32, activation  = 'relu'),
    keras.layers.Dense(2)
])
model.compile(optimizer = 'adam', 
              loss = keras.losses.SparseCategoricalCrossentropy(from_logits = True),
             metrics = ['accuracy'])
x = np.column_stack((train_df.x.values, train_df.y.values))

C:\Users\newta\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
model.fit(x, train_df.color.values, batch_size = 4, epochs = 8)

Epoch 1/8
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.7580 - loss: 0.4930
Epoch 2/8
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.8965 - loss: 0.2800
Epoch 3/8
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.9350 - loss: 0.1809
Epoch 4/8
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 3s 2ms/step - accuracy: 0.9690 - loss: 0.1109
Epoch 5/8
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9730 - loss: 0.0974
Epoch 6/8
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9885 - loss: 0.0522
Epoch 7/8
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9870 - loss: 0.0465
Epoch 8/8
1000/1000 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9835 - loss: 0.0549


#### Evaluating the test data

In [29]:
test_df = pd.read_csv('./quadratic/data/test.csv')

In [30]:
test_df.head()


,x,y,color
0,-0.709694,1.902953,0.0
1,1.358063,2.923887,0.0
2,-0.684174,2.125850,0.0
3,3.190384,11.425374,0.0
4,-1.225524,3.178416,0.0


In [31]:
test_x = np.column_stack((test_df.x.values, test_df.y.values))

In [32]:
model.evaluate(test_x, test_df.color.values)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0173


[0.01733783632516861, 1.0]

# Cluster

In [33]:
os.listdir('.')

['.ipynb_checkpoints',
 'clusters',
 'clusters_two_categories',
 'complex',
 'Images',
 'linear',
 'Neural Net.ipynb',
 'quadratic']

In [34]:
os.listdir('./clusters/data')

['test.csv', 'train.csv']

In [35]:
train_df = pd.read_csv('./clusters/data/train.csv')

In [36]:
np.random.shuffle(train_df.values)

In [37]:
train_df.color.unique()

array(['red', 'blue', 'green', 'teal', 'orange', 'purple'], dtype=object)

In [38]:
color_dict = {'red':0, 'blue':1, 'green': 2, 'teal':3, 'orange':4, 'purple':5}

##### Color: apply and change it to numerical value

In [39]:
train_df['color'] = train_df.color.apply(lambda x: color_dict[x])

In [40]:
train_df.color.unique()

array([0, 1, 2, 3, 4, 5])

In [41]:
model = keras.Sequential([
    keras.layers.Dense(32,input_shape=(2,), activation = 'relu'), 
   # keras.layers.Dropout(0.2), : Does not help
     keras.layers.Dense(32, activation  = 'relu'),
    keras.layers.Dense(6)
])
model.compile(optimizer = 'adam', 
              loss = keras.losses.SparseCategoricalCrossentropy(from_logits = True),
             metrics = ['accuracy'])
x = np.column_stack((train_df.x.values, train_df.y.values))

C:\Users\newta\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [42]:
model.fit(x, train_df.color.values, batch_size = 4, epochs = 8)

Epoch 1/8
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.7955 - loss: 0.5818
Epoch 2/8
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9503 - loss: 0.1716
Epoch 3/8
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9603 - loss: 0.1232
Epoch 4/8
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9648 - loss: 0.1029
Epoch 5/8
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9682 - loss: 0.0907
Epoch 6/8
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9688 - loss: 0.0835
Epoch 7/8
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.9707 - loss: 0.0797
Epoch 8/8
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 4s 2ms/step - accuracy: 0.9725 - loss: 0.0758


#### Evaluating the test data

In [43]:
test_df = pd.read_csv('./clusters/data/test.csv')

In [44]:
test_df.head()

,x,y,color
0,-0.173868,1.381852,red
1,-0.724148,1.883008,red
2,-0.423915,1.408297,red
3,-0.650162,1.296067,red
4,0.957933,0.633111,red


In [45]:
test_x = np.column_stack((test_df.x.values, test_df.y.values))
test_df['color'] = test_df.color.apply(lambda x: color_dict[x])
model.evaluate(test_x, test_df.color.values)

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9658 - loss: 0.1002


[0.10016903281211853, 0.965833306312561]

In [46]:
# Get the raw logits
logits = model.predict(np.array([[0,3]]))

# Convert logits to probabilities
probabilities = tf.nn.softmax(logits).numpy()

print(np.round(probabilities))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step
[[0. 0. 0. 0. 0. 1.]]


# 2 Category : Predicting two outcomes (Complex)

#### labels will be like first six will be for color, last 3 for marker

In [58]:
os.listdir('.')

['.ipynb_checkpoints',
 'clusters',
 'clusters_two_categories',
 'complex',
 'Images',
 'linear',
 'Neural Net.ipynb',
 'quadratic']

In [79]:
os.listdir('./clusters_two_categories/')

['data', 'figure.png', 'network_clusters_2.py']

In [80]:
train_df = pd.read_csv('./clusters_two_categories/data/train.csv')

In [81]:
train_df.head()

,x,y,color,marker
0,-0.765775,1.006452,red,^
1,0.229016,0.094320,red,^
2,0.189987,1.589141,red,^
3,0.580085,0.520488,red,^
4,0.292287,0.317852,red,^


In [82]:
one_hot_color = pd.get_dummies(train_df.color).values.astype(int)
one_hot_marker = pd.get_dummies(train_df.marker).values.astype(int)

In [83]:
one_hot_color

array([[0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 1, 0],
       [0, 0, 0, 0, 1, 0],
       ...,
       [0, 0, 0, 1, 0, 0],
       [0, 0, 0, 1, 0, 0],
       [0, 0, 0, 1, 0, 0]], shape=(6000, 6))

In [84]:
labels = np.concatenate((one_hot_color, one_hot_marker), axis = 1)

In [85]:
type(labels)

numpy.ndarray

In [86]:
model = keras.Sequential([
    keras.layers.Dense(32,input_shape=(2,), activation = 'relu'), 
    keras.layers.Dense(32, activation = 'relu'),
    keras.layers.Dense(9, activation = 'sigmoid')
])

model.compile(
    optimizer='adam', 
    loss=keras.losses.BinaryCrossentropy(from_logits=False), # False because of 'sigmoid' above
    metrics=['binary_accuracy'] # Use this for multi-label!
)

x = np.column_stack((train_df.x.values, train_df.y.values))

C:\Users\newta\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


#### Here, seed = 42 ensures that the shuffling logic order is same

In [87]:
np.random.RandomState(seed=42).shuffle(x)
np.random.RandomState(seed=42).shuffle(labels)

## How to know WHEN TO STOPP OVERFITTING (High number of epoch, etc.)

2. How to find it: Validation Data
Right now, you are only looking at Training Accuracy. This can be deceptive. A model can get 100% accuracy just by memorizing the spreadsheet, but it might fail in the real world.

The Fix: Use the validation_split argument in model.fit.

Python
model.fit(x, labels, 
          batch_size=4, 
          epochs=50, # Set it higher than you think you need
          validation_split=0.2, # Reserve 20% of data to "test" the model
          shuffle=True)
Now, Keras will show you two sets of numbers: loss/accuracy and val_loss/val_accuracy.

Watch the val_loss.

If loss keeps going down but val_loss starts to go up, you have passed the ideal epoch.

3. The Pro Way: Early Stopping
Instead of guessing the number of epochs, you can tell Keras to stop automatically when the model stops improving. This is the most "ideal" way to handle epochs.

Python
from tensorflow.keras.callbacks import EarlyStopping

# Stop training when validation loss hasn't improved for 3 epochs
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

model.fit(x, labels, 
          epochs=100, # Set a high number
          validation_split=0.2, 
          callbacks=[early_stop])

In [88]:
model.fit(x, labels, 
          batch_size=4, 
          epochs=30, # Set it higher than you think you need
          validation_split=0.2, # Reserve 20% of data to "test" the model
          shuffle=True)

Epoch 1/30
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 6s 4ms/step - binary_accuracy: 0.8680 - loss: 0.2897 - val_binary_accuracy: 0.9205 - val_loss: 0.1728
Epoch 2/30
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - binary_accuracy: 0.9495 - loss: 0.1290 - val_binary_accuracy: 0.9566 - val_loss: 0.1021
Epoch 3/30
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - binary_accuracy: 0.9740 - loss: 0.0798 - val_binary_accuracy: 0.9761 - val_loss: 0.0697
Epoch 4/30
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - binary_accuracy: 0.9805 - loss: 0.0597 - val_binary_accuracy: 0.9807 - val_loss: 0.0550
Epoch 5/30
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - binary_accuracy: 0.9848 - loss: 0.0495 - val_binary_accuracy: 0.9863 - val_loss: 0.0433
Epoch 6/30
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - binary_accuracy: 0.9858 - loss: 0.0435 - val_binary_accuracy: 0.9860 - val_loss: 0.0402
Epoch 7/30
1200/1200 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - binary_accuracy: 0.9870 - loss: 0.0384 - val_binary_accuracy: 0.9929 - val_loss:

1. Does from_logits=True pick the activation?Yes, exactly. When you set from_logits=True, the loss function looks at its own name to decide which math to apply to your raw output:If the loss is BinaryCrossentropy: It applies Sigmoid internally.If the loss is CategoricalCrossentropy: It applies Softmax internally.It effectively "injects" that math right at the very last second before calculating the error.


2. Do you need to mention activation in the last layer?No. In fact, if you set from_logits=True, you should not put an activation like sigmoid or softmax in your last Dense layer.The "Clean" Way: Leave the last layer as Dense(9) (which is linear/raw). Use from_logits=True.The "Double-Activation" Danger: If you put activation='sigmoid' AND from_logits=True, Keras will take your probabilities and run the Sigmoid math on them again. This will mess up your gradients and make your model train very slowly or poorly.Rule of Thumb:activation='sigmoid' $\rightarrow$ from_logits=FalseNo activation $\rightarrow$ from_logits=True (Preferred for stability)


3. What is metrics=['binary_accuracy'] for?You are right—it is primarily for visualization and monitoring.For the Model: The "Loss" is the only thing the model actually uses to update its weights. The model doesn't care about "Accuracy"; it only cares about making that Loss number as small as possible.For You: "Loss" is hard for humans to interpret. If I tell you my loss is 0.0256, you don't really know if that's good or bad. If I tell you my accuracy is 99%, you instantly know the model is performing well.How it calculates the percentage:For binary_accuracy, it takes your output, rounds it to the nearest whole number (0 or 1), and compares it to the label.Logit $> 0 \rightarrow$ Prediction is 1.Logit $\leq 0 \rightarrow$ Prediction is 0.It then counts how many of these "bits" you got right across your 9 categories and gives you the average.Final "Cheat Sheet" for your code:ComponentYour ChoiceWhy?Last LayerDense(9) (No activation)Allows for maximum numerical stability.LossBinaryCrossentropy(from_logits=True)Treats 9 outputs as independent Yes/No tasks.MetricBinaryAccuracy()Gives you "partial credit" for each correct tag.

# METRICES 

1. metrics=['accuracy'] (The "Guesser")When you use the string 'accuracy', Keras does not have a single formula. Instead, it guesses which accuracy to use based on your loss function and your label shape.If your labels are 2D (like yours): Keras often guesses CategoricalAccuracy.The "All-or-Nothing" Rule: CategoricalAccuracy only gives you a point if the index of the maximum value in your prediction matches the index of the maximum value in your label.Why it failed you: Since you have two "1s" in your labels (one for color, one for marker), there is no single "maximum index." The math looks for one winner, finds two, and decides you got 0% correct.
  
 2. metrics=['binary_accuracy'] (The "Bit-Checker")When you specify binary_accuracy, you are telling Keras: "Treat every one of my 9 outputs as an independent True/False question."The "Partial Credit" Rule: It looks at every single bit. If you have 9 outputs and you get 8 of them right, you get 88.8% accuracy for that row.How it works with Logits: It internally checks if your output is $> 0$ (for from_logits=True) or $> 0.5$ (for from_logits=False). If it matches the 0 or 1 in your label, it's a "hit."3.

 
 
 Comparison Table Metric
 
 How it sees your 9 labelsReal-world Analogy'accuracy' (defaulting to Categorical)"Did you get the entire row of 9 perfectly right as a single category?"A multiple-choice test where you can only pick one answer.'binary_accuracy'"For these 9 individual traits, how many did you identify correctly?"A checklist where you get a point for every box you check correctly.

### Testing the example

In [89]:
test_df = pd.read_csv('./clusters_two_categories/data/test.csv')
test_x = np.column_stack((test_df.x.values, test_df.y.values))

In [90]:
test_one_hot_color = pd.get_dummies(test_df.color).values
test_one_hot_marker = pd.get_dummies(test_df.marker).values

In [91]:
test_labels = np.concatenate((test_one_hot_color, test_one_hot_marker), axis=1)

In [92]:
model.evaluate(test_x, test_labels)

38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - binary_accuracy: 0.9929 - loss: 0.0157


[0.01569814234972, 0.9928703904151917]

# COMPLEX FIGURE

In [94]:
os.listdir('./complex/')

['data', 'figure.png', 'network_complex.py']

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

img = mpimg.imread('./complex/figure')
plt.imshow(img)
plt.axis('off')  # hides axes
plt.show()